# Backprop practice 4: forward and backward by hand

Four networks in the style of practice 2 and 3. For each one, write `forward` (store what backward needs) and
`backward` (return a dict of gradients keyed by parameter name, e.g. `{"W1": dW1, "gamma": dgamma}`). NumPy only,
no autograd.

Each test checks the forward loss against a fixed value and compares your gradients with numerical ones (central
differences on every entry of every parameter). Write the shape next to every line.

In [ ]:
import numpy as np

sigmoid = lambda x: 1 / (1 + np.exp(-x))


def grad_check(net, X, Y, grads, params, h=1e-6, tol=1e-6):
    """Central differences on every entry of every parameter; compares with your analytic grads."""
    for name in params:
        P = getattr(net, name)
        num = np.zeros_like(P)
        for idx in np.ndindex(P.shape):
            old = P[idx]
            P[idx] = old + h
            lp = net.forward(X, Y)
            P[idx] = old - h
            lm = net.forward(X, Y)
            P[idx] = old
            num[idx] = (lp - lm) / (2 * h)
        g = grads[name]
        assert g.shape == P.shape, f"{name}: grad shape {g.shape} != param shape {P.shape}"
        rel = np.linalg.norm(num - g) / max(1e-12, np.linalg.norm(num) + np.linalg.norm(g))
        assert rel < tol, f"{name}: relative error {rel:.2e}"
        print(f"{name}: relative error {rel:.1e}  ok")


def data(seed, B=6, D=5, O=3, classes=False):
    rng = np.random.RandomState(seed)
    X = rng.randn(B, D)                                                  # [B, D]
    Y = rng.randint(O, size=B) if classes else rng.randn(B, O)           # [B] labels or [B, O] targets
    return X, Y

## Problem 1: product of a layer with its own row sum (warm-up)

`H2 = H1_scaled * S` where `S` is built from `H1_scaled` itself: the gradient reaches `H1_scaled` along two paths.

In [ ]:
class GateNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
        self.gamma = np.random.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = tanh(Z1)                                 [B, H]
        3. H1_scaled = H1 * gamma                        [B, H]
        4. S = sum(H1_scaled, axis=1, keepdims=True)     [B, 1]
        5. H2 = H1_scaled * S                            [B, H]
        6. out = H2 @ W2                                 [B, O]
        7. loss = mean((out - Y_target)^2)               scalar, mean over all elements
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "gamma": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
np.random.seed(0)
net = GateNet(5, 8, 3)
X, Y = data(1)
loss = net.forward(X, Y)
assert np.isclose(loss, 0.8517886211, atol=1e-8), f"forward: loss {loss}, expected 0.8517886211"
grad_check(net, X, Y, net.backward(), ["W1", "W2", "gamma"])
print("ok")

## Problem 2: batch norm, then softmax cross-entropy

The classic. `mu` and `var` are computed over the batch, so every row's gradient depends on every other row.

In [ ]:
class BatchNormNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
        self.gamma = np.random.randn(hidden_dim)
        self.beta = np.random.randn(hidden_dim)
        self.eps = 1e-5

    def forward(self, X, y):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = softplus(Z1) = log(1 + exp(Z1))          [B, H]
        3. mu = mean(H1, axis=0)                         [H]
        4. Hc = H1 - mu                                  [B, H]
        5. var = mean(Hc^2, axis=0)                      [H]
        6. Hn = Hc / sqrt(var + eps)                     [B, H]
        7. H2 = Hn * gamma + beta                        [B, H]
        8. logits = H2 @ W2                              [B, C]
        9. softmax with the max trick, cross-entropy with integer labels y [B], mean over the batch
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "gamma": ..., "beta": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
np.random.seed(0)
net = BatchNormNet(5, 8, 3)
X, y = data(1, classes=True)
loss = net.forward(X, y)
assert np.isclose(loss, 1.6646711686, atol=1e-8), f"forward: loss {loss}, expected 1.6646711686"
grad_check(net, X, y, net.backward(), ["W1", "W2", "gamma", "beta"])
print("ok")

## Problem 3: max normalization, SiLU and a residual

`max` sends its gradient to the argmax only. The residual means `Hn` gets gradient from two places.

In [ ]:
class MaxResNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) / np.sqrt(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
        self.alpha = np.random.randn(hidden_dim)

    def forward(self, X, y):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = relu(Z1)                                 [B, H]
        3. M = max(H1, axis=1, keepdims=True)            [B, 1]
        4. Hn = H1 / (M + 1e-6)                          [B, H]
        5. Z2 = Hn @ W2                                  [B, H]
        6. H2 = Z2 * sigmoid(Z2)                         [B, H]  (SiLU)
        7. H3 = H2 * alpha + Hn                          [B, H]  (residual)
        8. logits = H3 @ W3                              [B, C]
        9. softmax with the max trick, cross-entropy with integer labels y [B], mean over the batch
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "W3": ..., "alpha": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
np.random.seed(0)
net = MaxResNet(5, 8, 3)
X, y = data(1, classes=True)
loss = net.forward(X, y)
assert np.isclose(loss, 1.3136987012, atol=1e-8), f"forward: loss {loss}, expected 1.3136987012"
grad_check(net, X, y, net.backward(), ["W1", "W2", "W3", "alpha"])
print("ok")

## Problem 4: fix the backward pass

The forward is correct. The backward has three bugs. Find and fix them; the gradient check tells you when you are done.

In [ ]:
class DebugNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
        self.beta = np.random.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        (THIS IS CORRECT)
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = sigmoid(Z1)                              [B, H]
        3. Hs = H1 + beta                                [B, H]
        4. H2 = Hs - mean(Hs, axis=1, keepdims=True)     [B, H]  (center each row)
        5. out = H2 @ W2                                 [B, O]
        6. loss = mean((out - Y_target)^2)               scalar, mean over all elements
        """
        self.X, self.Y = X, Y_target
        self.Z1 = X @ self.W1
        self.H1 = sigmoid(self.Z1)
        self.Hs = self.H1 + self.beta
        self.H2 = self.Hs - self.Hs.mean(axis=1, keepdims=True)
        self.out = self.H2 @ self.W2
        return np.mean((self.out - Y_target) ** 2)

    def backward(self):
        B = self.X.shape[0]
        dout = 2 * (self.out - self.Y) / B
        dW2 = self.H2.T @ dout
        dH2 = dout @ self.W2.T
        dHs = dH2
        dbeta = dHs.sum(axis=0)
        dZ1 = dHs * self.H1
        dW1 = self.X.T @ dZ1
        return {"W1": dW1, "W2": dW2, "beta": dbeta}

In [ ]:
np.random.seed(0)
net = DebugNet(5, 8, 3)
X, Y = data(1)
loss = net.forward(X, Y)
assert np.isclose(loss, 1.1315817559, atol=1e-8), f"forward: loss {loss}, expected 1.1315817559"
grad_check(net, X, Y, net.backward(), ["W1", "W2", "beta"])
print("ok")